# Multiclass Classification with mlcore

This notebook demonstrates how to train and evaluate models on multiclass classification tasks.

Topics covered:

- Multiclass datasets
- Random Forest training
- XGBoost training
- Multiclass evaluation metrics
- Model comparison

In [1]:
import pandas as pd

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

import mlcore.classification

from mlcore.core.registry import MODEL_REGISTRY
from mlcore.core.trainer import Trainer

from mlcore.evaluation.classification import (
    evaluate_classification,
)

In [2]:
trainer = Trainer(
    registry=MODEL_REGISTRY,
)

In [3]:
X, y = make_classification(
    n_samples=2000,
    n_features=50,
    n_informative=25,
    n_redundant=10,
    n_classes=4,
    n_clusters_per_class=2,
    random_state=42,
)

print(X.shape)
print(y.shape)

(2000, 50)
(2000,)


In [4]:
pd.Series(y).value_counts().sort_index()

0    498
1    497
2    508
3    497
Name: count, dtype: int64

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42,
)

In [6]:
rf_result = trainer.fit(
    algorithm="random_forest",
    X=X_train,
    y=y_train,
    n_estimators=200,
    random_state=42,
)

In [7]:
rf_predictions = trainer.predict(
    rf_result,
    X_test,
)

In [8]:
rf_metrics = evaluate_classification(
    y_true=y_test,
    y_pred=rf_predictions,
)

rf_metrics

{'accuracy': 0.725,
 'balanced_accuracy': 0.724590017825312,
 'precision_macro': 0.7303437738731857,
 'precision_micro': 0.725,
 'precision_weighted': 0.7303715049656226,
 'recall_macro': 0.724590017825312,
 'recall_micro': 0.725,
 'recall_weighted': 0.725,
 'f1_macro': 0.7259614584308707,
 'f1_micro': 0.725,
 'f1_weighted': 0.7261904566830009,
 'mcc': 0.6340661838007515}

In [9]:
xgb_result = trainer.fit(
    algorithm="xgb_classifier",
    X=X_train,
    y=y_train,
    n_estimators=200,
    max_depth=6,
    random_state=42,
    verbosity=0,
)

In [10]:
xgb_predictions = trainer.predict(
    xgb_result,
    X_test,
)

In [11]:
xgb_metrics = evaluate_classification(
    y_true=y_test,
    y_pred=xgb_predictions,
)

xgb_metrics

{'accuracy': 0.745,
 'balanced_accuracy': 0.7449881164587047,
 'precision_macro': 0.7459340481242003,
 'precision_micro': 0.745,
 'precision_weighted': 0.7463270673368357,
 'recall_macro': 0.7449881164587047,
 'recall_micro': 0.745,
 'recall_weighted': 0.745,
 'f1_macro': 0.745086155674391,
 'f1_micro': 0.745,
 'f1_weighted': 0.7452868092691622,
 'mcc': 0.6602597321142195}

In [12]:
comparison = pd.DataFrame(
    [
        rf_metrics,
        xgb_metrics,
    ],
    index=[
        "Random Forest",
        "XGBoost",
    ],
)

comparison

,accuracy,balanced_accuracy,precision_macro,precision_micro,precision_weighted,recall_macro,recall_micro,recall_weighted,f1_macro,f1_micro,f1_weighted,mcc
Random Forest,0.725,0.724590,0.730344,0.725,0.730372,0.724590,0.725,0.725,0.725961,0.725,0.726190,0.634066
XGBoost,0.745,0.744988,0.745934,0.745,0.746327,0.744988,0.745,0.745,0.745086,0.745,0.745287,0.660260


In [13]:
comparison.sort_values(
    by="f1_macro",
    ascending=False,
)

,accuracy,balanced_accuracy,precision_macro,precision_micro,precision_weighted,recall_macro,recall_micro,recall_weighted,f1_macro,f1_micro,f1_weighted,mcc
XGBoost,0.745,0.744988,0.745934,0.745,0.746327,0.744988,0.745,0.745,0.745086,0.745,0.745287,0.660260
Random Forest,0.725,0.724590,0.730344,0.725,0.730372,0.724590,0.725,0.725,0.725961,0.725,0.726190,0.634066


In [14]:
rf_result.backend.get_metadata()

{'classes': array([0, 1, 2, 3]), 'n_features': 50}

In [15]:
MODEL_REGISTRY.filter(
    task="classification",
)

[AlgorithmSpec(backend='lightgbm', task='classification', name='lgbm_classifier', runner=<function make_classifier_runner.<locals>.runner at 0x7f3ba82df6a0>, backend_cls=<class 'mlcore.classification.backend.ClassificationBackend'>, aliases=(), default_params={}, search_space={}, tags=('classification', 'lightgbm', 'tree', 'boosting'), supports_cv=False, supports_proba=True, description=None),
 AlgorithmSpec(backend='sklearn', task='classification', name='logistic_regression', runner=<function make_classifier_runner.<locals>.runner at 0x7f3ba78fade0>, backend_cls=<class 'mlcore.classification.backend.ClassificationBackend'>, aliases=(), default_params={}, search_space={}, tags=('classification', 'sklearn'), supports_cv=False, supports_proba=True, description=None),
 AlgorithmSpec(backend='sklearn', task='classification', name='random_forest', runner=<function make_classifier_runner.<locals>.runner at 0x7f3ba7919da0>, backend_cls=<class 'mlcore.classification.backend.ClassificationBacke

## Understanding Macro vs Weighted Metrics

Macro metrics:
- Treat all classes equally.

Weighted metrics:
- Weight classes according to their frequency.

Macro metrics are generally preferred when class balance is important.

## Summary

In this notebook we:

- Created a multiclass dataset.
- Trained Random Forest and XGBoost classifiers.
- Evaluated multiclass performance.
- Compared models using macro and weighted metrics.
- Explored metadata stored by mlcore.

This workflow demonstrates the multiclass classification capabilities of the framework.